# CNN+LSTM Modeling — CRISP-DM Phase 4
## Credit Card Fraud Detection (Sparkov Dataset)

**Objectif :** Construire, entraîner et évaluer CNN+LSTM+Attention

**Papers :**
- PeerJ 2024 (Wu & Chen) — CNN+LSTM+Attention supérieur à CNN/LSTM seuls
- Journal of Big Data 2025 — SMOTE+CNN+Attention F1=90.60%
- arXiv:2506.02703 — SMOTE après split uniquement (anti-leakage)
- arXiv:2502.00201 — F1-score métrique principale pour données déséquilibrées
- Fusion 2025 — EarlyStopping + ModelCheckpoint best practices

In [ ]:
# Cell 1 — Imports
import os
os.chdir('/mnt/c/Users/phili/OneDrive/Desktop/PFE-PROJET1.1/hybrid_ai_platform')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.keras

DATA_PROC = Path('data/processed')
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)
Path('notebooks/figures').mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print('CNN+LSTM Modeling — CRISP-DM Phase 4')
print('=' * 50)
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Cell 2 — Chargement des données Phase 3
print('[1] Loading preprocessed sequences...')

X_train = np.load(DATA_PROC / 'X_train.npy')
y_train = np.load(DATA_PROC / 'y_train.npy')
X_val   = np.load(DATA_PROC / 'X_val.npy')
y_val   = np.load(DATA_PROC / 'y_val.npy')
X_test  = np.load(DATA_PROC / 'X_test.npy')
y_test  = np.load(DATA_PROC / 'y_test.npy')

with open(DATA_PROC / 'feature_info.json') as f:
    feature_info = json.load(f)

WINDOW_SIZE = feature_info['window_size']
N_FEATURES  = feature_info['n_features']

print(f'X_train : {X_train.shape} | fraud: {y_train.mean()*100:.2f}%')
print(f'X_val   : {X_val.shape}   | fraud: {y_val.mean()*100:.2f}%')
print(f'X_test  : {X_test.shape}  | fraud: {y_test.mean()*100:.2f}%')
print(f'window_size={WINDOW_SIZE} | n_features={N_FEATURES}')

In [ ]:
# Cell 3 — SMOTE sur train uniquement (arXiv:2506.02703)
print('[2] SMOTE — train set only (anti-leakage rule arXiv:2506.02703)')
print('-' * 50)
print('CRITICAL: SMOTE appliqué APRÈS split — jamais avant')
print(f'  Before SMOTE — Normal: {int((y_train==0).sum()):,} | Fraud: {int((y_train==1).sum()):,}')

# Reshape pour SMOTE : (N, 10, 27) → (N, 270)
N_train = X_train.shape[0]
X_train_2d = X_train.reshape(N_train, -1)

smote = SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE, n_jobs=-1)
X_train_sm, y_train_sm = smote.fit_resample(X_train_2d, y_train)

# Reshape retour : (N_new, 270) → (N_new, 10, 27)
X_train_sm = X_train_sm.reshape(-1, WINDOW_SIZE, N_FEATURES).astype(np.float32)
y_train_sm = y_train_sm.astype(np.float32)

print(f'  After SMOTE  — Normal: {int((y_train_sm==0).sum()):,} | Fraud: {int((y_train_sm==1).sum()):,}')
print(f'  New fraud rate: {y_train_sm.mean()*100:.2f}%')
print(f'  X_train_sm shape: {X_train_sm.shape}')

In [ ]:
# Cell 4 — Architecture CNN+LSTM+Attention (PeerJ 2024 + Journal of Big Data 2025)
print('[3] Building CNN+LSTM+Attention architecture')
print('-' * 50)

def build_cnn_lstm_attention(window_size, n_features,
                              conv_filters=64, kernel_size=3,
                              lstm_units=128, dense_units=64,
                              dropout_rate=0.3, learning_rate=1e-3):
    """
    CNN+LSTM+Attention for fraud detection.
    
    Architecture based on:
    - PeerJ 2024 (Wu & Chen): CNN+LSTM+Attention surpasses CNN/LSTM alone
    - Journal of Big Data 2025: Attention mechanism for imbalanced fraud data
    
    Input  : (batch, window_size=10, n_features=27)
    Output : (batch, 1) — fraud probability
    """
    inputs = keras.Input(shape=(window_size, n_features), name='input')
    
    # CNN block — extract local patterns (Wu & Chen 2024)
    x = layers.Conv1D(conv_filters, kernel_size,
                      activation='relu', padding='same', name='conv1d')(inputs)
    x = layers.MaxPooling1D(pool_size=2, name='maxpool')(x)
    x = layers.BatchNormalization(name='bn_conv')(x)
    
    # LSTM block — capture temporal dependencies
    x = layers.LSTM(lstm_units, return_sequences=True, name='lstm')(x)
    x = layers.Dropout(dropout_rate, name='dropout_lstm')(x)
    
    # Attention mechanism — focus on most relevant timesteps
    # (Journal of Big Data 2025)
    attention_scores = layers.Dense(1, activation='tanh', name='attn_dense')(x)
    attention_weights = layers.Softmax(axis=1, name='attn_softmax')(attention_scores)
    x = layers.Multiply(name='attn_multiply')([x, attention_weights])
    x = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name='attn_sum')(x)
    
    # Classification head
    x = layers.Dense(dense_units, activation='relu', name='dense1')(x)
    x = layers.Dropout(dropout_rate, name='dropout_dense')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='output')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='CNN_LSTM_Attention')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    
    return model

model = build_cnn_lstm_attention(
    window_size=WINDOW_SIZE,
    n_features=N_FEATURES,
)
model.summary()

In [ ]:
# Cell 5 — Visualisation architecture
print('[4] Architecture visualization')

# Paramètres du modèle
total_params = model.count_params()
print(f'  Total parameters: {total_params:,}')
print(f'  Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

# Plot architecture sous forme de tableau
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

layers_info = [
    ['Layer', 'Type', 'Output Shape', 'Params'],
    ['input', 'Input', f'(None, {WINDOW_SIZE}, {N_FEATURES})', '0'],
    ['conv1d', 'Conv1D(64, k=3)', f'(None, {WINDOW_SIZE}, 64)', str(64*3*N_FEATURES+64)],
    ['maxpool', 'MaxPooling1D', f'(None, {WINDOW_SIZE//2}, 64)', '0'],
    ['bn_conv', 'BatchNorm', f'(None, {WINDOW_SIZE//2}, 64)', '256'],
    ['lstm', 'LSTM(128)', f'(None, {WINDOW_SIZE//2}, 128)', str(4*(128*(64+128)+128))],
    ['attention', 'Attention', f'(None, 128)', 'trainable'],
    ['dense1', 'Dense(64)+ReLU', '(None, 64)', str(128*64+64)],
    ['output', 'Dense(1)+Sigmoid', '(None, 1)', '65'],
]

table = ax.table(
    cellText=layers_info[1:],
    colLabels=layers_info[0],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2)

# Header couleur
for j in range(4):
    table[0, j].set_facecolor('#1a1a2e')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Couleurs par type
colors = ['#E3F2FD', '#E3F2FD', '#E3F2FD', '#E3F2FD',
          '#FFF3E0', '#FCE4EC', '#E8F5E9', '#E8F5E9']
for i, color in enumerate(colors):
    for j in range(4):
        table[i+1, j].set_facecolor(color)

ax.set_title('CNN+LSTM+Attention Architecture\n(PeerJ 2024 + Journal of Big Data 2025)',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('notebooks/figures/09_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Callbacks (Fusion 2025 best practices)
print('[5] Setting up callbacks (Fusion 2025)')

BEST_MODEL_PATH = str(MODELS_DIR / 'best_cnn_lstm_fraud.keras')

callbacks = [
    # Sauvegarde du meilleur modèle (val_f1 ou val_auc)
    ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    # Arrêt si pas d'amélioration pendant 5 epochs
    EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Réduction du learning rate si plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
]

# Class weights pour gérer déséquilibre résiduel
fraud_ratio = y_train_sm.mean()
class_weight = {
    0: 1.0,
    1: (1 - fraud_ratio) / fraud_ratio
}
print(f'  Class weights: {class_weight}')
print(f'  Best model path: {BEST_MODEL_PATH}')

In [ ]:
# Cell 7 — Entraînement avec MLflow tracking
print('[6] Training CNN+LSTM+Attention with MLflow tracking')
print('-' * 50)

mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('fraud_detection_cnn_lstm')

EPOCHS     = 30
BATCH_SIZE = 256

with mlflow.start_run(run_name='CNN_LSTM_Attention_v1') as run:
    # Log hyperparamètres
    mlflow.log_params({
        'model_type': 'CNN_LSTM_Attention',
        'window_size': WINDOW_SIZE,
        'n_features': N_FEATURES,
        'conv_filters': 64,
        'lstm_units': 128,
        'dense_units': 64,
        'dropout_rate': 0.3,
        'learning_rate': 1e-3,
        'batch_size': BATCH_SIZE,
        'epochs_max': EPOCHS,
        'smote_strategy': 0.1,
        'dataset': 'Sparkov_CreditCard_2019_2020',
        'paper_ref': 'PeerJ_2024_CNN_LSTM_Attention',
    })
    
    # Entraînement
    history = model.fit(
        X_train_sm, y_train_sm,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )
    
    # Log métriques finales
    best_epoch = np.argmax(history.history['val_auc'])
    mlflow.log_metrics({
        'best_epoch': int(best_epoch),
        'best_val_auc': float(history.history['val_auc'][best_epoch]),
        'best_val_loss': float(history.history['val_loss'][best_epoch]),
        'train_auc_final': float(history.history['auc'][-1]),
    })
    
    RUN_ID = run.info.run_id
    print(f'\n  MLflow run_id: {RUN_ID}')
    print(f'  Best epoch: {best_epoch+1} | val_auc: {history.history["val_auc"][best_epoch]:.4f}')

In [ ]:
# Cell 8 — Courbes d'apprentissage
print('[7] Learning curves')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'],     label='Train Loss', color='#2196F3')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#F44336')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Binary Crossentropy')
axes[0].legend()
axes[0].axvline(best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best epoch {best_epoch+1}')
axes[0].legend()

# AUC
axes[1].plot(history.history['auc'],     label='Train AUC', color='#2196F3')
axes[1].plot(history.history['val_auc'], label='Val AUC',   color='#F44336')
axes[1].set(title='AUC-ROC', xlabel='Epoch', ylabel='AUC')
axes[1].axhline(0.90, color='green', linestyle='--', alpha=0.5, label='Target: 0.90')
axes[1].legend()

# Precision / Recall
axes[2].plot(history.history['precision'], label='Train Precision', color='#2196F3')
axes[2].plot(history.history['recall'],    label='Train Recall',    color='#FF9800')
axes[2].plot(history.history['val_precision'], label='Val Precision', color='#2196F3', linestyle='--')
axes[2].plot(history.history['val_recall'],    label='Val Recall',    color='#FF9800', linestyle='--')
axes[2].set(title='Precision & Recall', xlabel='Epoch', ylabel='Score')
axes[2].legend(fontsize=8)

plt.suptitle('CNN+LSTM+Attention — Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figures/10_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9 — Évaluation complète (arXiv:2502.00201)
print('[8] Evaluation — F1-score + AUC-ROC + Confusion Matrix')
print('-' * 50)

# Charger le meilleur modèle
best_model = keras.models.load_model(BEST_MODEL_PATH)

# Prédictions
y_pred_proba = best_model.predict(X_test, batch_size=512, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

# Métriques
f1        = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
auc_roc   = roc_auc_score(y_test, y_pred_proba)
avg_prec  = average_precision_score(y_test, y_pred_proba)

# RMSE des résidus (pour FeedbackLoop)
rmse = float(np.sqrt(np.mean((y_test - y_pred_proba)**2)))

print(f'  F1-score  : {f1:.4f}    (target > 0.80)')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  AUC-ROC   : {auc_roc:.4f}  (target > 0.90)')
print(f'  Avg Prec  : {avg_prec:.4f}')
print(f'  RMSE      : {rmse:.4f}    (FeedbackLoop threshold: 0.15)')
print()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

# Log dans MLflow
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({
        'test_f1': f1,
        'test_precision': precision,
        'test_recall': recall,
        'test_auc_roc': auc_roc,
        'test_avg_precision': avg_prec,
        'test_rmse': rmse,
    })

In [ ]:
# Cell 10 — Visualisations évaluation
print('[9] Evaluation plots')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
axes[0].set(title='Confusion Matrix', xlabel='Predicted', ylabel='Actual')

# Precision-Recall curve
prec_curve, rec_curve, thresholds = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(rec_curve, prec_curve, color='#2196F3', lw=2,
             label=f'AP = {avg_prec:.3f}')
axes[1].axhline(y_test.mean(), color='red', linestyle='--',
                label=f'Baseline: {y_test.mean():.3f}')
axes[1].set(title='Precision-Recall Curve',
            xlabel='Recall', ylabel='Precision')
axes[1].legend()

# Distribution des probabilités
axes[2].hist(y_pred_proba[y_test==0], bins=50, alpha=0.7,
             color='#2196F3', label='Normal', density=True)
axes[2].hist(y_pred_proba[y_test==1], bins=50, alpha=0.7,
             color='#F44336', label='Fraud',  density=True)
axes[2].axvline(0.5, color='black', linestyle='--', label='Threshold=0.5')
axes[2].set(title='Predicted Probability Distribution',
            xlabel='P(Fraud)', ylabel='Density')
axes[2].legend()

plt.suptitle(f'CNN+LSTM+Attention — Test Evaluation | F1={f1:.3f} | AUC={auc_roc:.3f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figures/11_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11 — Sauvegarde modèle final
print('[10] Saving final model')

# Sauvegarder en format Keras
final_model_path = MODELS_DIR / 'cnn_lstm_fraud_final.keras'
best_model.save(str(final_model_path))

# Sauvegarder les métriques
model_metrics = {
    'model_type': 'CNN_LSTM_Attention',
    'dataset': 'Sparkov_CreditCard_2019_2020',
    'window_size': WINDOW_SIZE,
    'n_features': N_FEATURES,
    'test_f1': round(f1, 4),
    'test_precision': round(precision, 4),
    'test_recall': round(recall, 4),
    'test_auc_roc': round(auc_roc, 4),
    'test_rmse': round(rmse, 4),
    'mlflow_run_id': RUN_ID,
    'paper_ref': 'PeerJ_2024_Wu_Chen_CNN_LSTM_Attention',
}

with open(MODELS_DIR / 'model_metrics.json', 'w') as f:
    json.dump(model_metrics, f, indent=2)

print(f'  Model saved: {final_model_path}')
print(f'  Metrics saved: models/model_metrics.json')
print(f'  Model size: {final_model_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# Cell 12 — Résumé Phase 4
print('=' * 60)
print('CNN+LSTM MODELING SUMMARY — CRISP-DM Phase 4 Complete')
print('=' * 60)
print(f"""
Architecture: CNN+LSTM+Attention
  Input  : ({WINDOW_SIZE}, {N_FEATURES})
  Conv1D : filters=64, kernel=3
  LSTM   : units=128, return_sequences=True
  Attn   : Softmax attention over timesteps
  Dense  : 64 → 1 (Sigmoid)

Training:
  SMOTE strategy=0.1 (train only — arXiv:2506.02703)
  Optimizer: Adam lr=1e-3
  Loss: Binary Crossentropy + class_weight
  Callbacks: EarlyStopping + ModelCheckpoint + ReduceLROnPlateau
  MLflow run_id: {RUN_ID}

Results:
  F1-score  : {f1:.4f}  (target > 0.80)
  Precision : {precision:.4f}
  Recall    : {recall:.4f}
  AUC-ROC   : {auc_roc:.4f}  (target > 0.90)
  RMSE      : {rmse:.4f}  (FeedbackLoop threshold: 0.15)

FeedbackLoop status:
  RMSE {'> 0.15 → FeedbackLoop WILL trigger' if rmse > 0.15 else '< 0.15 → FeedbackLoop OK'}

Next Step: CRISP-DM Phase 5 — Evaluation
  → notebooks/04_agents_testing.ipynb
""")